In [1]:
import os
import sys
import pathlib as pl
sys.path.append(str(pl.Path().cwd().parent))
os.chdir(str(pl.Path().cwd().parent))
import json
from src.binance_data_collector import BinanceDataCollector
from src.data_preprocessor import DataPreprocessor


In [2]:
print(pl.Path.cwd())

/Users/franciscojara/Documents/lstm_crypto_dkk


In [3]:
with open('validation/healthy_dates.json', 'r') as f:
    config_dates = json.load(f)

In [4]:
config_dates

{'symbols': ['BTC-USD', 'ETH-USD'],
 'days_back': 1460,
 'total_unique_dates': 1461,
 'train': {'start': '2022-09-02', 'end': '2025-03-01', 'count': 912},
 'validation': {'start': '2025-03-02', 'end': '2025-11-30', 'count': 274},
 'test': {'start': '2025-12-01', 'end': '2026-09-01', 'count': 275}}

In [6]:
SYMBOLS = ['BTC-USD']
collector = BinanceDataCollector()
interval = '5m'
crypto_raw_data = collector.fetch_crypto_data(symbols = SYMBOLS, interval = interval, 
                                              days_back=config_dates['days_back'])

INFO:root:Fetching Binance data for 1 symbols...
INFO:root:Interval: 1m, Days back: 1460
INFO:root:Successfully collected data for 1 symbols
INFO:root:Total records: 2102320
INFO:root:Date range: 2022-09-19 16:06:00 to 2026-09-18 16:05:00


In [8]:
crypto_raw_data

,Date,Symbol,Open,High,Low,Close,Volume,Quote_volume,Count,Taker_buy_volume,Taker_buy_quote_volume
0,2022-09-19 16:06:00,BTC-USD,19112.60,19113.71,19065.86,19070.51,253.05800,4.830817e+06,3695,92.39699,1.763484e+06
1,2022-09-19 16:07:00,BTC-USD,19070.51,19144.44,19063.55,19140.04,321.27705,6.139256e+06,5592,203.32615,3.885579e+06
2,2022-09-19 16:08:00,BTC-USD,19140.04,19170.00,19135.25,19169.97,348.26761,6.670502e+06,5193,194.61615,3.727863e+06
3,2022-09-19 16:09:00,BTC-USD,19169.96,19171.91,19148.98,19164.59,196.41264,3.763588e+06,3209,113.04270,2.166220e+06
4,2022-09-19 16:10:00,BTC-USD,19163.27,19188.88,19156.17,19185.71,225.34372,4.321034e+06,3788,107.83218,2.067696e+06
...,...,...,...,...,...,...,...,...,...,...,...
2102315,2026-09-18 16:01:00,BTC-USD,80705.44,80737.45,80671.53,80737.45,18.17139,1.466611e+06,4501,5.32658,4.299202e+05
2102316,2026-09-18 16:02:00,BTC-USD,80737.44,80790.01,80695.03,80790.00,11.50646,9.290401e+05,4002,6.55930,5.296302e+05
2102317,2026-09-18 16:03:00,BTC-USD,80790.00,80818.40,80691.29,80704.98,25.51047,2.059535e+06,5131,8.74266,7.056990e+05
2102318,2026-09-18 16:04:00,BTC-USD,80704.98,80830.00,80694.97,80827.41,12.18517,9.841933e+05,5400,5.54870,4.481946e+05


In [7]:
preprocessor = DataPreprocessor()

In [9]:
INPUT_STUDY_FEATURES = ['Close', 'Volume', 'Quote_volume', 'Count']
processed_data = preprocessor.scale_features(crypto_raw_data.set_index(['Date', 'Symbol']), scaler = 'minmax')
processed_data

,,Open,High,Low,Close,Volume,Quote_volume,Count,Taker_buy_volume,Taker_buy_quote_volume
Date,Symbol,,,,,,,,,
2022-09-19 16:06:00,BTC-USD,0.032538,0.032256,0.032459,0.032158,0.043053,0.018591,0.017339,0.026120,0.011897
2022-09-19 16:07:00,BTC-USD,0.032158,0.032533,0.032438,0.032786,0.054660,0.023626,0.026240,0.057478,0.026213
2022-09-19 16:08:00,BTC-USD,0.032786,0.032764,0.033087,0.033057,0.059252,0.025671,0.024368,0.055016,0.025149
2022-09-19 16:09:00,BTC-USD,0.033057,0.032781,0.033211,0.033008,0.033416,0.014484,0.015058,0.031956,0.014614
2022-09-19 16:10:00,BTC-USD,0.032996,0.032935,0.033276,0.033199,0.038338,0.016629,0.017775,0.030483,0.013949
...,...,...,...,...,...,...,...,...,...,...
2026-09-18 16:01:00,BTC-USD,0.589432,0.589154,0.589493,0.589722,0.003092,0.005644,0.021121,0.001506,0.002900
2026-09-18 16:02:00,BTC-USD,0.589722,0.589629,0.589706,0.590197,0.001958,0.003575,0.018779,0.001854,0.003573
2026-09-18 16:03:00,BTC-USD,0.590197,0.589886,0.589672,0.589428,0.004340,0.007926,0.024077,0.002471,0.004761


In [10]:
config_dates

{'symbols': ['BTC-USD', 'ETH-USD'],
 'days_back': 1460,
 'total_unique_dates': 1461,
 'train': {'start': '2022-09-02', 'end': '2025-03-01', 'count': 912},
 'validation': {'start': '2025-03-02', 'end': '2025-11-30', 'count': 274},
 'test': {'start': '2025-12-01', 'end': '2026-09-01', 'count': 275}}

In [12]:
# Build training and validation splits here from the API-fetched dataframe.
lstm_data = preprocessor.prepare_lstm_data(processed_data.reset_index(), symbols = SYMBOLS,
                                           sequence_length = 5,
                                           target_col = 'Close',
                                           prediction_horizon=1,
                                           date_splits =  {
                                               'train': config_dates['train'],
                                               'validation': config_dates['validation'],
                                               'test': config_dates['test']
                                           },
                                           input_study_features = INPUT_STUDY_FEATURES
                                           )
    

INFO:root:Using the following Close
INFO:root:Target Column is Close
INFO:root: LSTM matrix transformation complete
INFO:root:   X_train: (1286310, 5, 4), y_train: (1286310,)
INFO:root:   train dates:      2022-09-19 to 2025-03-01
INFO:root:   X_val:   (393121, 5, 4), y_val:   (393121,)
INFO:root:   val dates:        2025-03-02 to 2025-11-30
INFO:root:   X_test:  (394561, 5, 4), y_test:  (394561,)
INFO:root:   test dates:       2025-12-01 to 2026-09-01


In [ ]:
print(lstm_data.keys())
lstm_data['dates_train']

dict_keys(['X_train', 'y_train', 'dates_train', 'X_val', 'y_val', 'dates_val', 'X_test', 'y_test', 'dates_test', 'feature_columns', 'target_col', 'sequence_length', 'prediction_horizon', 'symbols'])


array(['2025-07-09T21:57:00.000000000', '2025-07-09T21:58:00.000000000',
       '2025-07-09T21:59:00.000000000', ...,
       '2026-01-04T23:58:00.000000000', '2026-01-04T23:59:00.000000000',
       '2026-01-05T00:00:00.000000000'], dtype='datetime64[ns]')

In [16]:
lstm_data['y_train']

array([0.6444022 , 0.64967092, 0.63348803, ..., 0.65356192, 0.65647156,
       0.6587826 ])

In [20]:
lstm_data['X_train']

array([[[0.64197965, 0.0043027 , 0.00933496],
        [0.648548  , 0.00179381, 0.00306472],
        [0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725]],

       [[0.648548  , 0.00179381, 0.00306472],
        [0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725],
        [0.6444022 , 0.00184823, 0.00343549]],

       [[0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725],
        [0.6444022 , 0.00184823, 0.00343549],
        [0.64967092, 0.00380341, 0.01002487]],

       ...,

       [[0.64767248, 0.00132155, 0.00362792],
        [0.65080248, 0.00274569, 0.00486695],
        [0.64332128, 0.00500196, 0.00487164],
        [0.65146277, 0.01255966, 0.00326184],
        [0.65366125, 0.00193892, 0.00529403]],

       [[0.65080248, 0.00274569, 0.00486695],
        [0.64

In [19]:
processed_data[['Returns', 'Volume', 'Count']].head(10)

,,Returns,Volume,Count
Date,Symbol,,,
2025-07-09 21:52:00,BTC-USD,0.641980,0.004303,0.009335
2025-07-09 21:53:00,BTC-USD,0.648548,0.001794,0.003065
2025-07-09 21:54:00,BTC-USD,0.653740,0.005945,0.010490
2025-07-09 21:55:00,BTC-USD,0.646750,0.000757,0.002009
2025-07-09 21:56:00,BTC-USD,0.648429,0.001146,0.000807
2025-07-09 21:57:00,BTC-USD,0.644402,0.001848,0.003435
2025-07-09 21:58:00,BTC-USD,0.649671,0.003803,0.010025
2025-07-09 21:59:00,BTC-USD,0.633488,0.005499,0.010917
2025-07-09 22:00:00,BTC-USD,0.666204,0.004648,0.014254
